# Example 1.1.1 / 1: Flat Sets

Companion notebook for **1.1.1 Flat vs Indexed Sets** in the AMPL lab Documentation, section "Flat Sets". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Flat Sets

A flat set is referred to by its name alone: `PROD`, `ARCS`. "Flat" doesn't mean
one-dimensional; its members may be single items or tuples.

| Form | Example | Meaning |
|---|---|---|
| Single members | `set PROD;` | product names |
| Pairs | `set ARCS dimen 2;` | any pairs |
| Restricted pairs | `set ARCS within NODES cross NODES;` | pairs of known nodes only |
| Subset | `set B within A;` | members must also be in `A` |

In `data.py` and `run.py`, `ampl` is an `AMPL` object that has already read `model.mod`. A set of
pairs is given from Python as a list of tuples, and `to_list()` returns it the same way; `size()`
counts the members:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set NODES;
set ARCS within NODES cross NODES;   # one set of pairs
set PROD;
set B within PROD;                    # every member must be a product

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["NODES"] = ["A", "B", "C", "D"]
ampl.set["ARCS"] = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D")]
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.set["B"] = ["bands", "plate"]

In [5]:
# run.py
print(ampl.set["ARCS"].to_list())
print(ampl.set["ARCS"].size())
print(ampl.set["B"].to_list())

[('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D')]
4
['bands', 'plate']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`. In a data file, each pair is written in
parentheses:

In [6]:
%%writefile data.dat
set NODES := A B C D;
set ARCS := (A,B) (A,C) (B,D) (C,D);
set PROD := bands coils plate;
set B := bands plate;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display ARCS, card(ARCS), B;

set ARCS := (A,B) (A,C) (B,D) (C,D);

card(ARCS) = 4

set B := bands plate;

